1. IMPORTS

In [1]:
import re
import json
import pdfplumber

from Extraction.pdf_loader import load_pdf
from Extraction.table_extractor import (
    extract_words,
    extract_characters
)

from parsing.row_parser import (
    build_rows,
    classify_rows
)

from parsing.geometry_parser import (
    classify_relative_position,
    extract_action_metadata
)

from parsing.task_parser import parse_task_row

from parsing.hierarchy_parser import (
    get_identifier,
    get_chapter,
    get_node_type,
    get_process_id,
    get_parent_task_id,
    get_children,
    has_children
)

2. Load ODF

In [2]:
pdf = load_pdf("../Data/Raw/dam_mvp_chapters.pdf")

print("Number of pages:", len(pdf.pages))

Number of pages: 66


3. Page Selection

In [3]:
page_number = 10  # change whenever needed

page = pdf.pages[page_number - 1]

words = extract_words(page)
chars = extract_characters(page)

print("Words:", len(words))
print("Characters:", len(chars))

Words: 373
Characters: 1819


4. Row Investigation

In [4]:
rows = build_rows(words)

for y in sorted(rows.keys()):

    line = " ".join(
        w["text"]
        for w in rows[y]
    )

    print(y, "->", line)

52 -> PSO 2.100: CLIENT RELATIONS & MISSIONS
67 -> SNVP/RDVP/AHVP/PEVP/PIVP/FIVP/ECVP
69 -> DAM Code No. PSO 2.110 Authority Codes
86 -> I Initiate/Originate
102 -> C Check and Verify
109 -> Process: 2.110 CLIENT RELATIONS & COMMUNICATIONS
117 -> R Review & Recommend
132 -> A Approve / Sign
146 -> December 2022
149 -> Date: ( i ) To be Informed
165 -> maeT
167 -> )GESP( SROTCERID
171 -> tnioP noisiviD
172 -> noitatnemelpmI
173 -> GDD
178 -> .tpeD
179 -> tnatsissA
183 -> reciffO
185 -> )desab-noigeR
186 -> -QH(
187 -> PV
189 -> ksaT
194 -> lacoF
195 -> /
196 -> stsilaicepS
199 -> lareneG-yraterceS
200 -> reganaM
201 -> reganaM rotceS rotceS
205 -> tpeD
206 -> tegduB
208 -> reganaM
211 -> /reganaM
213 -> ACTION margorP
215 -> lesnuoC
216 -> tnemesrubsiD
217 -> /
221 -> rotanidrooC
222 -> FO
226 -> gnitroppuS
228 -> denrecnoC denrecnoC
229 -> 1
230 -> MOCSPO
233 -> srebmeM
235 -> reganaM
236 -> lanoigeR lanoigeR
237 -> yrtnuoC rotceriD DRAOB
238 -> troppuS troppuS
243 -> /
245 -> rotceS P

5. Print only numbered rows

In [5]:
import re

for y in sorted(rows.keys()):

    line = " ".join(
        w["text"]
        for w in rows[y]
    )

    if re.match(r"^\d", line):
        print(line)

1
2.110 CLIENT RELATIONS AND COMMUNICATIONS
2.111
2
2.112 Routine communication with Executing Agencies / PIUs I A ( i ) ( i ) ( i ) ( i ) ( i ) ( i )
3
2.113 Critical communication with Executing Agencies / PIUs I ( i ) R ( i ) ( i ) A R ( i ) ( i )
2.114 Communication from Regional Offices to Government on ( i )
4
2.115
2.116
5
2.117 Exceptional communication by Senior Management to
2.118 Communication with Co-Financiers of projects I ( i ) R ( i ) ( i ) A ( i ) ( i )
2.119
10


Survey all numbered lines in the DAM

In [6]:
import re

PATTERN = re.compile(r"^\d")

for page_num, page in enumerate(pdf.pages):

    rows = build_rows(
        extract_words(page)
    )

    print(f"\n===== PAGE {page_num+1} =====")

    for y in sorted(rows.keys()):

        line = " ".join(
            w["text"]
            for w in rows[y]
        )

        if PATTERN.match(line):
            print(line)


===== PAGE 1 =====
1. PROGRAMMING FOR COUNTRY & REGIONAL
1

===== PAGE 2 =====
3reciffO
2
6
5
1 noitargetnI &
8
01
9
1.110 Country Strategy Papers (CSP), Regional Integration Strategy Papers (RISP), and other programming documents
1.111 Selection of the Team
1.111.1
1.111.2
2

===== PAGE 3 =====
3reciffO
2
6
5
1 noitargetnI &
8
01
9
1.110 Country Strategy Papers (CSP), Regional Integration Strategy Papers (RISP), and other programming documents
1.112 Diagnostic Note
1.112.1
1.112.2
1.113 Completion Report
1.113.1
8
1.113.2
3

===== PAGE 4 =====
3reciffO
2
6
5
1 noitargetnI &
8
01
9
1.110 Country Strategy Papers (CSP), Regional Integration Strategy Papers (RISP), and other programming documents
1.114 New Strategy Paper
1.114.1 Preparation, review and approval of New
1.114.2
1.115 Other Programming Documents 15
1.115.1 Interim CSP/RISP; Country Brief; or JCAS Follow the respective process for new CSP/RISP (See 1.114.1 / 1.114.2)
1.115.2 Updated CSP/RISP (‘Extension’) Follow the respecti

Survey all identifiers only

In [7]:
import re

ID_PATTERN = re.compile(r"^\d+(?:\.\d+)+")

for page_num, page in enumerate(pdf.pages):

    rows = build_rows(
        extract_words(page)
    )

    print(f"\n===== PAGE {page_num+1} =====")

    for y in sorted(rows.keys()):

        line = " ".join(
            w["text"]
            for w in rows[y]
        )

        m = ID_PATTERN.match(line)

        if m:
            print(m.group())


===== PAGE 1 =====

===== PAGE 2 =====
1.110
1.111
1.111.1
1.111.2

===== PAGE 3 =====
1.110
1.112
1.112.1
1.112.2
1.113
1.113.1
1.113.2

===== PAGE 4 =====
1.110
1.114
1.114.1
1.114.2
1.115
1.115.1
1.115.2
1.116
1.116.1
1.116.2

===== PAGE 5 =====
1.110
1.117
1.117.1
1.117.2

===== PAGE 6 =====

===== PAGE 7 =====

===== PAGE 8 =====

===== PAGE 9 =====

===== PAGE 10 =====
2.110
2.111
2.112
2.113
2.114
2.115
2.116
2.117
2.118
2.119

===== PAGE 11 =====

===== PAGE 12 =====
2.120
2.121
2.122
2.123
2.124
2.125
2.126

===== PAGE 13 =====

===== PAGE 14 =====
2.210
2.211
2.211.1
2.211.2
2.211.3
2.211.4
2.212
2.212.1
2.213
2.213.1
2.213.2
2.213.3

===== PAGE 15 =====

===== PAGE 16 =====
2.220
2.220
2.221
2.221.1
2.221.2
2.221.3
2.222
2.222.1
2.222.2
2.223
2.223.1

===== PAGE 17 =====
2.220
2.223.2
2.223.3
2.224
2.224.1
2.224.2
2.224.3
2.224.4
2.225
2.225.1
2.225.2

===== PAGE 18 =====

===== PAGE 19 =====

===== PAGE 20 =====

===== PAGE 21 =====
2.310
2.311
2.312.1
2.312.2
2.313
2.314


Print rows within a Y-range

In [8]:
page = pdf.pages[23]

rows = build_rows(
    extract_words(page)
)

for y in sorted(rows.keys()):

    if 320 <= y <= 360:

        line = " ".join(
            w["text"]
            for w in rows[y]
        )

        print(y, "->", line)

324 -> 2.325 Portfolio Performance Review:
336 -> 2.325.1
338 -> C o u n try / R e g i on a l Portfolio Performance Review
341 -> 10 10 I10 10 10 10
343 -> C1 C1 ( i ) R1 R1 R2 ( i ) ( i ) A1 ( i ) ( i )
346 -> 1 0
348 -> (C P P R / R P P R )


Print nearby characters around one action

In [9]:
page = pdf.pages[23]

chars = extract_characters(page)

for c in chars:

    if c["text"] != "R":
        continue

    print("\nBASE:")
    print(c)

    base_x = c["x0"]

    print("\nNEARBY:")

    for candidate in chars:

        distance = candidate["x0"] - base_x

        if distance < 0:
            continue

        if distance > 25:
            continue

        if abs(candidate["top"] - c["top"]) > 15:
            continue

        print(
            repr(candidate["text"]),
            "x=", round(candidate["x0"],1),
            "top=", round(candidate["top"],1),
            "size=", round(candidate["size"],1)
        )

    break


BASE:
{'matrix': (1, 0, 0, 1, 359.69, 533.74), 'fontname': 'ABCDEF+TimesNewRomanPS-BoldMT', 'adv': 7.191120000000001, 'upright': True, 'x0': 359.69, 'y0': 531.58864, 'x1': 366.88112, 'y1': 541.54864, 'width': 7.191120000000012, 'height': 9.959999999999923, 'size': 9.959999999999923, 'mcid': None, 'tag': None, 'object_type': 'char', 'page_number': 24, 'ncs': 'DeviceGray', 'text': 'R', 'stroking_color': (0,), 'non_stroking_color': (0.0,), 'top': 53.77137000000005, 'bottom': 63.73136999999997, 'doctop': 13746.131599999997}

NEARBY:
'R' x= 359.7 top= 53.8 size= 10.0
'D' x= 366.9 top= 53.8 size= 10.0
'V' x= 374.1 top= 53.8 size= 10.0
'P' x= 381.3 top= 53.8 size= 10.0


Test extract_action_metadata()

In [10]:
page = pdf.pages[23]

chars = extract_characters(page)

for c in chars:

    if c["text"] not in ["I", "C", "R", "A"]:
        continue

    result = extract_action_metadata(
        c,
        chars
    )

    if (
        result["authority_modifier"]
        or result["note_references"]
    ):
        print(result)

SUPER: / x= 366.0 top= 249.4
SUPER: / x= 366.0 top= 249.4
{'action': 'R', 'authority_modifier': '/', 'note_references': []}
{'action': 'R', 'authority_modifier': '/', 'note_references': []}
{'action': 'I', 'authority_modifier': '/', 'note_references': []}
{'action': 'R', 'authority_modifier': '21', 'note_references': []}
{'action': 'A', 'authority_modifier': '1', 'note_references': []}
SUPER: 9 x= 205.5 top= 302.5
{'action': 'C', 'authority_modifier': None, 'note_references': ['9']}
SUPER: 9 x= 205.5 top= 302.5
{'action': 'R', 'authority_modifier': None, 'note_references': ['9']}
SUPER: 9 x= 295.7 top= 302.5
{'action': 'I', 'authority_modifier': None, 'note_references': ['9']}
SUPER: 9 x= 404.9 top= 302.5
{'action': 'C', 'authority_modifier': None, 'note_references': ['9']}
SUPER: 9 x= 476.0 top= 302.5
{'action': 'R', 'authority_modifier': '12', 'note_references': ['9']}
SUPER: 9 x= 507.2 top= 302.5
{'action': 'R', 'authority_modifier': '2', 'note_references': ['9']}
{'action': 'A', 'a

Print all superscripts detected

In [11]:
for page_num, page in enumerate(pdf.pages):

    chars = extract_characters(page)

    for c in chars:

        if c["text"] not in ["I", "C", "R", "A"]:
            continue

        result = extract_action_metadata(
            c,
            chars
        )

        if result["note_references"]:

            print(
                "PAGE",
                page_num + 1,
                result
            )

SUPER: 8 x= 507.0 top= 255.7
PAGE 2 {'action': 'I', 'authority_modifier': None, 'note_references': ['8']}
SUPER: 1 x= 577.0 top= 291.1
PAGE 2 {'action': 'A', 'authority_modifier': None, 'note_references': ['1']}
SUPER: 1 x= 577.0 top= 291.1
SUPER: 0 x= 577.0 top= 288.1
PAGE 2 {'action': 'C', 'authority_modifier': None, 'note_references': ['10']}
SUPER: 8 x= 507.0 top= 255.7
PAGE 3 {'action': 'I', 'authority_modifier': None, 'note_references': ['8']}
SUPER: 1 x= 577.0 top= 291.1
PAGE 3 {'action': 'A', 'authority_modifier': None, 'note_references': ['1']}
SUPER: 1 x= 577.0 top= 291.1
SUPER: 0 x= 577.0 top= 288.1
PAGE 3 {'action': 'C', 'authority_modifier': None, 'note_references': ['10']}
SUPER: 4 x= 385.2 top= 367.8
PAGE 3 {'action': 'R', 'authority_modifier': None, 'note_references': ['4']}
SUPER: 1 x= 545.5 top= 367.8
PAGE 3 {'action': 'C', 'authority_modifier': None, 'note_references': ['1']}
SUPER: 1 x= 545.5 top= 367.8
SUPER: 1 x= 548.5 top= 367.8
PAGE 3 {'action': 'R', 'authority_

Test task parser

In [12]:
examples = [

    "2.324.1 Completion Evaluation Terms of Reference I C R2 A1 ( i )",

    "2.324.2 Project Completion Report (PCR) ( i ) ( i ) R1 R2 ( i ) ( i ) ( i ) A1 ( i )"
]

for row in examples:

    print("\nROW:")
    print(row)

    print("\nPARSED:")
    print(parse_task_row(row))


ROW:
2.324.1 Completion Evaluation Terms of Reference I C R2 A1 ( i )

PARSED:
{'task_id': '2.324.1', 'description': 'Completion Evaluation Terms of Reference', 'actions': [{'code': 'I', 'authority_level': None, 'meaning': None}, {'code': 'C', 'authority_level': None, 'meaning': None}, {'code': 'R', 'authority_level': 2, 'meaning': 'Review and clear for higher authority'}, {'code': 'A', 'authority_level': 1, 'meaning': 'Validate or endorse'}, {'code': 'i', 'modifier': None, 'meaning': 'To Be Informed'}]}

ROW:
2.324.2 Project Completion Report (PCR) ( i ) ( i ) R1 R2 ( i ) ( i ) ( i ) A1 ( i )

PARSED:
{'task_id': '2.324.2', 'description': 'Project Completion Report (PCR)', 'actions': [{'code': 'i', 'modifier': None, 'meaning': 'To Be Informed'}, {'code': 'i', 'modifier': None, 'meaning': 'To Be Informed'}, {'code': 'R', 'authority_level': 1, 'meaning': 'Review and recommend'}, {'code': 'R', 'authority_level': 2, 'meaning': 'Review and clear for higher authority'}, {'code': 'i', 'modi

Print all detected task IDs

In [13]:
for page_num, page in enumerate(pdf.pages):

    rows = build_rows(
        extract_words(page)
    )

    for y in sorted(rows.keys()):

        line = " ".join(
            w["text"]
            for w in rows[y]
        )

        m = re.match(
            r"^\d+\.\d+\.\d+",
            line
        )

        if m:
            print(m.group())

1.111.1
1.111.2
1.112.1
1.112.2
1.113.1
1.113.2
1.114.1
1.114.2
1.115.1
1.115.2
1.116.1
1.116.2
1.117.1
1.117.2
2.211.1
2.211.2
2.211.3
2.211.4
2.212.1
2.213.1
2.213.2
2.213.3
2.221.1
2.221.2
2.221.3
2.222.1
2.222.2
2.223.1
2.223.2
2.223.3
2.224.1
2.224.2
2.224.3
2.224.4
2.225.1
2.225.2
2.312.1
2.312.2
2.322.1
2.322.2
2.322.3
2.322.4
2.323.1
2.323.2
2.323.3
2.324.1
2.324.2
2.325.1
2.325.2
2.411.1
2.411.2
2.411.3
2.411.4
2.411.5
2.412.1
2.412.2
2.413.1
2.413.2
2.413.3
2.413.4
2.413.5
2.422.1
2.422.2
2.423.1
2.423.2
2.423.3
2.431.1
2.431.2
2.431.3
2.431.4
2.432.1
2.432.2
2.432.3
2.432.4
2.433.1
2.433.2
2.433.3
2.433.4
2.511.1
2.511.2
2.512.1
2.512.2
2.512.3
2.513.1
2.513.2
2.513.3
2.514.1
2.514.2
2.515.1
2.515.2
2.521.1
2.521.2
2.521.3
2.521.4
2.521.5
2.522.1
2.522.2
2.522.3
3.683.1
3.683.2
3.683.3
3.683.4


Detect special cases (2.111, 2.311, etc.)

In [14]:
for page_num, page in enumerate(pdf.pages):

    rows = build_rows(
        extract_words(page)
    )

    for y in sorted(rows.keys()):

        line = " ".join(
            w["text"]
            for w in rows[y]
        )

        m = re.match(
            r"^\d+\.\d{3}",
            line
        )

        if m:
            print(
                "PAGE",
                page_num+1,
                "->",
                line
            )

PAGE 2 -> 1.110 Country Strategy Papers (CSP), Regional Integration Strategy Papers (RISP), and other programming documents
PAGE 2 -> 1.111 Selection of the Team
PAGE 2 -> 1.111.1
PAGE 2 -> 1.111.2
PAGE 3 -> 1.110 Country Strategy Papers (CSP), Regional Integration Strategy Papers (RISP), and other programming documents
PAGE 3 -> 1.112 Diagnostic Note
PAGE 3 -> 1.112.1
PAGE 3 -> 1.112.2
PAGE 3 -> 1.113 Completion Report
PAGE 3 -> 1.113.1
PAGE 3 -> 1.113.2
PAGE 4 -> 1.110 Country Strategy Papers (CSP), Regional Integration Strategy Papers (RISP), and other programming documents
PAGE 4 -> 1.114 New Strategy Paper
PAGE 4 -> 1.114.1 Preparation, review and approval of New
PAGE 4 -> 1.114.2
PAGE 4 -> 1.115 Other Programming Documents 15
PAGE 4 -> 1.115.1 Interim CSP/RISP; Country Brief; or JCAS Follow the respective process for new CSP/RISP (See 1.114.1 / 1.114.2)
PAGE 4 -> 1.115.2 Updated CSP/RISP (‘Extension’) Follow the respective process for new CSP/RISP (See 1.114.1 / 1.114.2)
PAGE 4 -

Count task-like identifiers

In [15]:
count = 0

for page_num, page in enumerate(pdf.pages):

    rows = build_rows(
        extract_words(page)
    )

    for y in sorted(rows.keys()):

        line = " ".join(
            w["text"]
            for w in rows[y]
        )

        if re.match(
            r"^\d+\.\d+\.\d+",
            line
        ):
            count += 1

print("Task IDs found:", count)

Task IDs found: 102


**Hierarchy tests**

In [16]:
import importlib
import parsing.hierarchy_parser

importlib.reload(parsing.hierarchy_parser)

<module 'parsing.hierarchy_parser' from 'c:\\Users\\ademm\\OneDrive\\Documents\\damaiagent\\parsing\\hierarchy_parser.py'>

In [17]:
from parsing.hierarchy_parser import (
    get_chapter,
    get_node_type,
    get_process_id,
    get_parent_task_id,
    get_children,
    has_children
)

In [18]:


examples = [
    "2.110",
    "2.111",
    "2.221",
    "2.221.1",
    "2.221.2",
    "2.324.2",
    "3.683",
    "3.683.4",
    "1.111",
    "1.111.1"
]

for identifier in examples:

    print("\nID:", identifier)

    print(
        "chapter:",
        get_chapter(identifier)
    )

    print(
        "type:",
        get_node_type(identifier)
    )

    print(
        "process:",
        get_process_id(identifier)
    )

    print(
        "parent:",
        get_parent_task_id(identifier)
    )


ID: 2.110
chapter: 2
type: process
process: 2.110
parent: None

ID: 2.111
chapter: 2
type: task
process: 2.110
parent: None

ID: 2.221
chapter: 2
type: task
process: 2.220
parent: None

ID: 2.221.1
chapter: 2
type: child_task
process: 2.220
parent: 2.221

ID: 2.221.2
chapter: 2
type: child_task
process: 2.220
parent: 2.221

ID: 2.324.2
chapter: 2
type: child_task
process: 2.320
parent: 2.324

ID: 3.683
chapter: 3
type: task
process: 3.680
parent: None

ID: 3.683.4
chapter: 3
type: child_task
process: 3.680
parent: 3.683

ID: 1.111
chapter: 1
type: task
process: 1.110
parent: None

ID: 1.111.1
chapter: 1
type: child_task
process: 1.110
parent: 1.111


In [19]:
all_ids = [
    "2.110",
    "2.111",
    "2.221",
    "2.221.1",
    "2.221.2",
    "2.221.3",
    "3.680",
    "3.683",
    "3.683.1",
    "3.683.2",
    "1.111",
    "1.111.1",
    "1.111.2"
]

for identifier in all_ids:

    if get_node_type(identifier) != "task":
        continue

    print(
        identifier,
        "children:",
        get_children(
            identifier,
            all_ids
        ),
        "has_children:",
        has_children(
            identifier,
            all_ids
        )
    )

2.111 children: [] has_children: False
2.221 children: ['2.221.1', '2.221.2', '2.221.3'] has_children: True
3.683 children: ['3.683.1', '3.683.2'] has_children: True
1.111 children: ['1.111.1', '1.111.2'] has_children: True


***First Attempt*** 
**Extraction of all Identifiers**

In [20]:
from parsing.hierarchy_parser import get_identifier

all_ids = []

for page in pdf.pages:

    words = extract_words(page)
    rows = build_rows(words)

    for y in sorted(rows.keys()):

        line = " ".join(
            w["text"]
            for w in rows[y]
        )

        identifier = get_identifier(line)

        if identifier:
            all_ids.append(identifier)

# remove duplicates
all_ids = sorted(
    set(all_ids)
)

print("Identifiers found:", len(all_ids))

Identifiers found: 275


**Identifier graph building**

In [21]:
from parsing.hierarchy_parser import (
    get_chapter,
    get_node_type,
    get_process_id,
    get_parent_task_id,
    get_children,
    has_children
)

nodes = {}

for identifier in all_ids:

    nodes[identifier] = {
        "id": identifier,
        "chapter": get_chapter(identifier),
        "node_type": get_node_type(identifier),
        "process_id": get_process_id(identifier),
        "parent_task_id":
            get_parent_task_id(identifier),
        "has_children":
            has_children(
                identifier,
                all_ids
            ),
        "children":
            get_children(
                identifier,
                all_ids
            )
    }

print(
    "Nodes built:",
    len(nodes)
)

Nodes built: 275


examples Inspection

In [22]:
examples = [
    "2.110",
    "2.111",
    "2.221",
    "2.221.1",
    "3.683",
    "3.683.4",
    "1.111"
]

for e in examples:

    print("\n================")
    print(nodes[e])


{'id': '2.110', 'chapter': '2', 'node_type': 'process', 'process_id': '2.110', 'parent_task_id': None, 'has_children': False, 'children': []}

{'id': '2.111', 'chapter': '2', 'node_type': 'task', 'process_id': '2.110', 'parent_task_id': None, 'has_children': False, 'children': []}

{'id': '2.221', 'chapter': '2', 'node_type': 'task', 'process_id': '2.220', 'parent_task_id': None, 'has_children': True, 'children': ['2.221.1', '2.221.2', '2.221.3']}

{'id': '2.221.1', 'chapter': '2', 'node_type': 'child_task', 'process_id': '2.220', 'parent_task_id': '2.221', 'has_children': False, 'children': []}

{'id': '3.683', 'chapter': '3', 'node_type': 'task', 'process_id': '3.680', 'parent_task_id': None, 'has_children': True, 'children': ['3.683.1', '3.683.2', '3.683.3', '3.683.4']}

{'id': '3.683.4', 'chapter': '3', 'node_type': 'child_task', 'process_id': '3.680', 'parent_task_id': '3.683', 'has_children': False, 'children': []}

{'id': '1.111', 'chapter': '1', 'node_type': 'task', 'process_i

**second task block parser test after the changes**

In [23]:
from parsing.task_block_parser import build_task_blocks

In [24]:
page = pdf.pages[23]  # page 24

rows = build_rows(
    extract_words(page)
)

blocks = build_task_blocks(rows)

for block in blocks:
    print("\n================")
    print(block)


2.324 Project / Programme Completion:

2.324.1 Completion Evaluation Terms of Reference I C R2 A1 ( i ) 9 I9 C9 9 9

2.324.2 Project Completion Report (PCR) ( i ) ( i ) R1 R2 ( i ) ( i ) ( i ) A1 ( i ) ( i ) ( i )

2.325 Portfolio Performance Review:

2.325.1 C o u n try / R e g i on a l Portfolio Performance Review 10 10 I10 10 10 10 C1 C1 ( i ) R1 R1 R2 ( i ) ( i ) A1 ( i ) ( i ) 1 0 (C P P R / R P P R )

2.325.2 Annual Portfolio Performance Review report R1 C1 ( i ) I R1 R1 ( i ) ( i ) ( i ) A1 ( i ) ( i )


**Task cleaner tests**

In [25]:
task_lookup = {}

for page_idx, page in enumerate(pdf.pages):

    rows = build_rows(
        extract_words(page)
    )

    blocks = build_task_blocks(rows)

    for block in blocks:

        identifier = get_identifier(block)

        if not identifier:
            continue

        task_lookup[identifier] = {
            "page": page_idx + 1,
            "text": block
        }

print(
    "Blocks found:",
    len(task_lookup)
)

Blocks found: 275


In [26]:
task_lookup["2.325.1"]
task_lookup["2.221.1"]
task_lookup["1.111"]
task_lookup["2.112"]

{'page': 10,
 'text': '2.112 Routine communication with Executing Agencies / PIUs I A ( i ) ( i ) ( i ) ( i ) ( i ) ( i ) R ( i ) 3'}

In [27]:
task_lookup["2.325.1"]

{'page': 24,
 'text': '2.325.1 C o u n try / R e g i on a l Portfolio Performance Review 10 10 I10 10 10 10 C1 C1 ( i ) R1 R1 R2 ( i ) ( i ) A1 ( i ) ( i ) 1 0 (C P P R / R P P R )'}

**METADATA TASK PARSER TESTS**

In [28]:
import importlib
import parsing.task_metadata_parser as tmp

importlib.reload(tmp)

<module 'parsing.task_metadata_parser' from 'c:\\Users\\ademm\\OneDrive\\Documents\\damaiagent\\parsing\\task_metadata_parser.py'>

In [29]:
from parsing.task_metadata_parser import *

In [30]:
text = task_lookup["2.112"]["text"]

clean_text = re.sub(
    r"^\d+(?:\.\d+)+",
    "",
    text
)

print(text)
print("------------")
print(clean_text)
print("------------")
print(NOTE_PATTERN.findall(clean_text))

2.112 Routine communication with Executing Agencies / PIUs I A ( i ) ( i ) ( i ) ( i ) ( i ) ( i ) R ( i ) 3
------------
 Routine communication with Executing Agencies / PIUs I A ( i ) ( i ) ( i ) ( i ) ( i ) ( i ) R ( i ) 3
------------
['3']


In [31]:
text = task_lookup["2.112"]["text"]

print(extract_actions(text))
print(extract_note_references(text))
print(extract_title(text))

['I', 'A', '( i )', '( i )', '( i )', '( i )', '( i )', '( i )', 'R', '( i )']
['3']
Routine communication with Executing Agencies / PIUs


In [32]:
text = task_lookup["2.325.1"]["text"]

print("ACTIONS:")
print(extract_actions(text))

print("\nNOTES:")
print(extract_note_references(text))

print("\nTITLE:")
print(extract_title(text))

ACTIONS:
['I', 'C1', 'C1', '( i )', 'R1', 'R1', 'R2', '( i )', '( i )', 'A1', '( i )', '( i )']

NOTES:
['10']

TITLE:
Country / Regional Portfolio Performance Review (CPPR / RPPR)


**Building enrichment**


In [33]:
for identifier, node in nodes.items():

    lookup = task_lookup.get(identifier)

    if not lookup:
        continue

    text = lookup["text"]

    node["page"] = lookup["page"]

    node["raw_text"] = text

    node["title"] = extract_title(text)

    node["actions"] = extract_actions(text)

    node["note_references"] = (
        extract_note_references(text)
    )

***Enrichement Tests***

In [34]:
examples = [
    "2.325.1",
    "2.112",
    "2.221.1",
    "1.111"
]

for e in examples:

    print("\n====================")
    print(json.dumps(
        nodes[e],
        indent=4
    ))


{
    "id": "2.325.1",
    "chapter": "2",
    "node_type": "child_task",
    "process_id": "2.320",
    "parent_task_id": "2.325",
    "has_children": false,
    "children": [],
    "page": 24,
    "raw_text": "2.325.1 C o u n try / R e g i on a l Portfolio Performance Review 10 10 I10 10 10 10 C1 C1 ( i ) R1 R1 R2 ( i ) ( i ) A1 ( i ) ( i ) 1 0 (C P P R / R P P R )",
    "title": "Country / Regional Portfolio Performance Review (CPPR / RPPR)",
    "actions": [
        "I",
        "C1",
        "C1",
        "( i )",
        "R1",
        "R1",
        "R2",
        "( i )",
        "( i )",
        "A1",
        "( i )",
        "( i )"
    ],
    "note_references": [
        "10"
    ]
}

{
    "id": "2.112",
    "chapter": "2",
    "node_type": "task",
    "process_id": "2.110",
    "parent_task_id": null,
    "has_children": false,
    "children": [],
    "page": 10,
    "raw_text": "2.112 Routine communication with Executing Agencies / PIUs I A ( i ) ( i ) ( i ) ( i ) ( i ) ( i

## Final SANITY TEST BEFORE MODELING ##

In [35]:
missing_titles = []

for node in nodes.values():
    if not node.get("title"):
        missing_titles.append(node["id"])

print(
    "Missing titles:",
    len(missing_titles)
)
missing_titles[:20]

Missing titles: 4


['2.126', '2.522', '3.520', '3.610']

In [36]:
for node in nodes.values():

    actions = node.get(
        "actions",
        []
    )

    for a in actions:

        if a.startswith("I") and len(a) > 1:
            print(
                node["id"],
                actions
            )

In [41]:
orphans = []

for node in nodes.values():

    if (
        node["node_type"] == "child_task"
        and node["parent_task_id"] not in nodes
    ):
        orphans.append(node["id"])

print(orphans)

['2.312.1', '2.312.2']


**Anomaly Invistigations**

In [38]:
for i in ['2.126', '2.522', '3.520', '3.610']:
    print("\n================")
    print(i)

    if i in nodes:
        print(json.dumps(
            nodes[i],
            indent=4
        ))


2.126
{
    "id": "2.126",
    "chapter": "2",
    "node_type": "task",
    "process_id": "2.120",
    "parent_task_id": null,
    "has_children": false,
    "children": [],
    "page": 12,
    "raw_text": "2.126 ( i )",
    "title": "",
    "actions": [
        "( i )"
    ],
    "note_references": []
}

2.522
{
    "id": "2.522",
    "chapter": "2",
    "node_type": "task",
    "process_id": "2.520",
    "parent_task_id": null,
    "has_children": true,
    "children": [
        "2.522.1",
        "2.522.2",
        "2.522.3"
    ],
    "page": 38,
    "raw_text": "2.522",
    "title": "",
    "actions": [],
    "note_references": []
}

3.520
{
    "id": "3.520",
    "chapter": "3",
    "node_type": "process",
    "process_id": "3.520",
    "parent_task_id": null,
    "has_children": false,
    "children": [],
    "page": 57,
    "raw_text": "3.520",
    "title": "",
    "actions": [],
    "note_references": []
}

3.610
{
    "id": "3.610",
    "chapter": "3",
    "node_type": "proc

In [39]:
for i in orphans:

    print("\n================")
    print(i)

    print(json.dumps(
        nodes[i],
        indent=4
    ))


2.312.1
{
    "id": "2.312.1",
    "chapter": "2",
    "node_type": "child_task",
    "process_id": "2.310",
    "parent_task_id": "2.312",
    "has_children": false,
    "children": [],
    "page": 21,
    "raw_text": "2.312.1 Signature of Financing Agreements for ADB or I C ( i ) A3 6 ( i ) ( i ) R2 ( i ) ( i ) ADF loans, grants, or guarantees",
    "title": "Signature of Financing Agreements for ADB or ADF loans, grants, or guarantees",
    "actions": [
        "I",
        "C",
        "( i )",
        "A3",
        "( i )",
        "( i )",
        "R2",
        "( i )",
        "( i )"
    ],
    "note_references": [
        "6"
    ]
}

2.312.2
{
    "id": "2.312.2",
    "chapter": "2",
    "node_type": "child_task",
    "process_id": "2.310",
    "parent_task_id": "2.312",
    "has_children": false,
    "children": [],
    "page": 21,
    "raw_text": "2.312.2 Signature of Financing Agreements for See DAM 16.100, 16.200, 16.300, and 16.400 technical cooperation funds / faciliti

In [40]:
print(
    extract_note_references(
        nodes["2.312.2"]["raw_text"]
    )
)

[]


In [42]:
print("2.312" in task_lookup)
print("2.312" in nodes)

False
False


In [43]:
missing_parents = set()

for node in nodes.values():

    if (
        node["node_type"] == "child_task"
        and node["parent_task_id"] not in nodes
    ):
        missing_parents.add(
            node["parent_task_id"]
        )

print(missing_parents)

{'2.312'}


In [44]:
for parent_id in missing_parents:

    nodes[parent_id] = {
        "id": parent_id,
        "chapter": get_chapter(parent_id),
        "node_type": "task",
        "process_id": get_process_id(parent_id),
        "parent_task_id": None,
        "has_children": True,
        "children": [],
        "page": None,
        "raw_text": "",
        "title": "",
        "actions": [],
        "note_references": [],
        "is_placeholder": True,
        "is_incomplete": True
    }

In [45]:
for node in nodes.values():
    node["children"] = []
    node["has_children"] = False

for node in nodes.values():

    parent = node["parent_task_id"]

    if parent and parent in nodes:

        nodes[parent]["children"].append(
            node["id"]
        )

        nodes[parent]["has_children"] = True

In [46]:
orphans = []

for node in nodes.values():

    if (
        node["node_type"] == "child_task"
        and node["parent_task_id"] not in nodes
    ):
        orphans.append(node["id"])

print("Orphans:", len(orphans))
print(orphans)

Orphans: 0
[]


In [48]:
import os

print(
    os.path.exists("../Data/Processed")
)

True


In [49]:
with open(
    "../Data/Processed/dam_structured.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        list(nodes.values()),
        f,
        indent=4,
        ensure_ascii=False
    )

print(
    "Saved",
    len(nodes),
    "nodes."
)

Saved 276 nodes.
